In [108]:
# init
from pathlib import Path
from importlib import reload
import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsPandas.helpers as tph
import toolsSync.main as tsm
import os
import boto3
from dotenv import load_dotenv
import pandas as pd
import copy

def pckgs_reload():
    reload(tgm)
    reload(tph)
    reload(tsm)

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"

process_state = tgm.load(DATA_DIR / 'process_state.json')
print(len(process_state))

241


In [1]:
load_dotenv()
bucket_name = os.environ["B2_BUCKET_NAME"]
session = boto3.session.Session()
s3 = session.client(
    service_name="s3",
    aws_access_key_id=os.environ["B2_KEY_ID"],
    aws_secret_access_key=os.environ["B2_APPLICATION_KEY"],
    endpoint_url=os.environ["B2_ENDPOINT"]
)

241


In [78]:
logger = tgl.initiate_logger('logger')

# read bucket

In [93]:
task_meta = {'scrape':{'dir':Path("data/raw/osm countries queries/")}, 'clean':{'dir':Path("data/cleaned/")}}
task_responses = {t:s3.list_objects_v2(Bucket=bucket_name, Prefix=task_meta[t]['dir'].as_posix()) for t in task_meta.keys()}
[len(res['Contents']) for task,res in task_responses.items()]

[20, 1]

In [118]:
sync_state = {k:v for k,v in task_meta.items()}
for task, res in task_responses.items():
    sync_state[task]['b2_objects'] = [obj['Key']  for obj in res['Contents']]
    sync_state[task]['b2_countries'] = tgm.deleteDuplicates([Path(obj['Key']).parent  for obj in res['Contents']])
    sync_state[task]['local_countries'] = [dir.name for dir in (ROOT / sync_state[task]['dir']).glob('*') if dir.is_dir()]
    sync_state[task]['b2_not_in_local'] = tgm.complement(
        tgm.deleteDuplicates([Path(obj).parent.name for obj in sync_state[task]['b2_countries']]),
        sync_state[task]['local_countries']
    )
    sync_state[task]['local_not_in_b2'] = tgm.complement(
        sync_state[task]['local_countries'],
        tgm.deleteDuplicates([Path(obj).parent.name for obj in sync_state[task]['b2_countries']])
    )
# get len
sync_state_resume = copy.deepcopy(sync_state)
for task,val in sync_state_resume.items():
    for k,v in sync_state_resume[task].items():
        if k in ['dir']: continue
        sync_state_resume[task][k] = len(v)

In [119]:
pd.DataFrame(sync_state_resume).T

,dir,b2_countries,local_countries,b2_not_in_local,local_not_in_b2,b2_objects
scrape,data\raw\osm countries queries,17,98,1,98,20
clean,data\cleaned,1,83,1,83,1


# download data

In [87]:
pckgs_reload()

In [102]:
to_download_countries = ['Armenia']
task = 'scrape'

In [103]:
tsm.donwload_country_data_from_bucket(to_download_countries, task_meta[task]['dir'], ROOT / task_meta[task]['dir'], s3, logger)

[INFO] : Total files found for bucket in 'data\raw\osm countries queries': 20
[INFO] : * Total of countries to download: 1
[INFO] : * Country Armenia (0/1) files found: 4
[INFO] :   * Skip existing file c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\raw\osm countries queries\Armenia\lvl_2_chunk_0_rawOSMRes.json
[INFO] :   * Skip existing file c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\raw\osm countries queries\Armenia\lvl_4_chunk_0_rawOSMRes.json
[INFO] :   * Skip existing file c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\raw\osm countries queries\Armenia\lvl_6_chunk_0_rawOSMRes.json
[INFO] :   * Skip existing file c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\raw\osm countries queries\Armenia\lvl_8_chunk_0_rawOSMRes.json
[INFO] : * Number of downloaded files: 0/4
